# 01 – Inspeção inicial

**Objetivo:** conhecer a estrutura do dataset *Most Streamed Spotify Songs 2023* e diagnosticar
problemas que serão tratados
no notebook 02. Nenhuma alteração é feita nos dados aqui.

In [63]:
import sys
sys.path.append("..")

%load_ext autoreload
%autoreload 2

import pandas as pd
import numpy as np

from src.funcoes import carregar_dados

pd.set_option("display.max_columns", None) # Não esconde as colunas que o pandas normalmente esconderia

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [64]:
df = pd.read_csv("../data/spotify-2023.csv")

### Caracteres perdidos nos nomes

Na comparação entre as leituras em UTF-8 e latin-1, alguns nomes apareceram com o símbolo `�`
(caractere de substituição, código Unicode `U+FFFD`) mesmo na leitura correta em UTF-8. Isso indica
que as letras acentuadas originais já foram perdidas na fonte do dataset. Abaixo, conto quantas
linhas das colunas de texto têm esse problema, para avaliar se ele afeta a análise.

In [65]:
for coluna in ["track_name", "artist(s)_name"]:
    n = df[coluna].str.contains("\ufffd", regex=False).sum()
    print(f"{coluna}: {n} linhas com caracteres perdidos")

track_name: 70 linhas com caracteres perdidos
artist(s)_name: 48 linhas com caracteres perdidos


**Resultado:** 70 linhas de `track_name` (~7%) e 48 de `artist(s)_name` (~5%) têm caracteres perdidos.
Também se observou que alguns nomes de artistas estão truncados (ex.: "Bad B", "Selena G").

**Impacto:** as hipóteses usam apenas colunas numéricas, portanto não são afetadas. O problema afeta
somente a contagem de artistas mais frequentes (análise descritiva), o que será tratado no notebook 02.
Fica registrado como limitação da qualidade dos dados para o artigo.

In [66]:
df = carregar_dados("../data/spotify-2023.csv")

Dados carregados: 953 linhas e 24 colunas.


In [67]:
df.head()

,track_name,artist(s)_name,artist_count,released_year,released_month,released_day,in_spotify_playlists,in_spotify_charts,streams,in_apple_playlists,in_apple_charts,in_deezer_playlists,in_deezer_charts,in_shazam_charts,bpm,key,mode,danceability_%,valence_%,energy_%,acousticness_%,instrumentalness_%,liveness_%,speechiness_%
0,Seven (feat. Latto) (Explicit Ver.),"Latto, Jung Kook",2,2023,7,14,553,147,141381703,43,263,45,10,826,125,B,Major,80,89,83,31,0,8,4
1,LALA,Myke Towers,1,2023,3,23,1474,48,133716286,48,126,58,14,382,92,C#,Major,71,61,74,7,0,10,4
2,vampire,Olivia Rodrigo,1,2023,6,30,1397,113,140003974,94,207,91,14,949,138,F,Major,51,32,53,17,0,31,6
3,Cruel Summer,Taylor Swift,1,2019,8,23,7858,100,800840817,116,207,125,12,548,170,A,Major,55,58,72,11,0,11,15
4,WHERE SHE GOES,Bad Bunny,1,2023,5,18,3133,50,303236322,84,133,87,15,425,144,A,Minor,65,23,80,14,63,11,6


In [68]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 953 entries, 0 to 952
Data columns (total 24 columns):
 #   Column                Non-Null Count  Dtype
---  ------                --------------  -----
 0   track_name            953 non-null    str  
 1   artist(s)_name        953 non-null    str  
 2   artist_count          953 non-null    int64
 3   released_year         953 non-null    int64
 4   released_month        953 non-null    int64
 5   released_day          953 non-null    int64
 6   in_spotify_playlists  953 non-null    int64
 7   in_spotify_charts     953 non-null    int64
 8   streams               953 non-null    str  
 9   in_apple_playlists    953 non-null    int64
 10  in_apple_charts       953 non-null    int64
 11  in_deezer_playlists   953 non-null    str  
 12  in_deezer_charts      953 non-null    int64
 13  in_shazam_charts      903 non-null    str  
 14  bpm                   953 non-null    int64
 15  key                   858 non-null    str  
 16  mode               

In [69]:
df.describe().T # O T é para transpor a tabela e facilitar a visualização

,count,mean,std,min,25%,50%,75%,max
artist_count,953.0,1.556139,0.893044,1.0,1.0,1.0,2.0,8.0
released_year,953.0,2018.238195,11.116218,1930.0,2020.0,2022.0,2022.0,2023.0
released_month,953.0,6.033578,3.566435,1.0,3.0,6.0,9.0,12.0
released_day,953.0,13.930745,9.201949,1.0,6.0,13.0,22.0,31.0
in_spotify_playlists,953.0,5200.124869,7897.608990,31.0,875.0,2224.0,5542.0,52898.0
in_spotify_charts,953.0,12.009444,19.575992,0.0,0.0,3.0,16.0,147.0
in_apple_playlists,953.0,67.812172,86.441493,0.0,13.0,34.0,88.0,672.0
in_apple_charts,953.0,51.908709,50.630241,0.0,7.0,38.0,87.0,275.0
in_deezer_charts,953.0,2.666317,6.035599,0.0,0.0,0.0,2.0,58.0
bpm,953.0,122.540399,28.057802,65.0,100.0,121.0,140.0,206.0


In [70]:
df.describe(exclude="number").T # Colunas não númericas

,count,unique,top,freq
track_name,953,943,Flowers,2
artist(s)_name,953,645,Taylor Swift,34
streams,953,949,156338624,2
in_deezer_playlists,953,348,0,24
in_shazam_charts,903,198,0,344
key,858,11,C#,120
mode,953,2,Major,550


## Diagnóstico

### Valores faltantes
Quais colunas têm valores faltantes, quantos e em que proporção?

In [71]:
faltantes = df.isna().sum()
faltantes = faltantes[faltantes > 0]

pd.DataFrame({
    "faltantes": faltantes,
    "percentual (%)": (faltantes / len(df) * 100).round(2),
})

,faltantes,percentual (%)
in_shazam_charts,50,5.25
key,95,9.97


**Resultado:** apenas `in_shazam_charts` (50; 5,25%) e `key` (95; 9,97%) têm valores faltantes.
`in_shazam_charts` será descartada (fora do escopo, que é só o Spotify). As variáveis de sucesso
e de características de áudio estão completas.

#### Investigação da coluna `key`
Os valores faltantes em `key` são realmente ausência de informação? Comparamos os valores
presentes com as 12 notas musicais possíveis.

In [72]:
notas = ["C", "C#", "D", "D#", "E", "F", "F#", "G", "G#", "A", "A#", "B"]

print(df["key"].value_counts(dropna=False))

presentes = df["key"].dropna().unique()
ausentes = [nota for nota in notas if nota not in presentes]
print("\nNotas que nunca aparecem na coluna:", ausentes)

key
C#     120
G       96
NaN     95
G#      91
F       89
B       81
D       81
A       75
F#      73
E       62
A#      57
D#      33
Name: count, dtype: int64

Notas que nunca aparecem na coluna: ['C']


**Resultado:** a nota C nunca aparece, e a quantidade de faltantes (95) é compatível com a
frequência das demais notas. Hipótese: as notas podem estar codificadas de 0 a 11 (0 = C) e o
valor 0 foi interpretado como vazio na construção do dataset. Talvez, para o notebook 02:
preencher os faltantes com "C", registrando a suposição.

**Hipótese:** Sorteamos 10 NaN (amostra aleatória usando random) da coluna "key" e pesquisamos a nota dessas músicas por fora. Se a maioria for "C", então a ideia de preencher os faltantes com "C" é válida. 

In [73]:
df[df["key"].isna()][["track_name", "artist(s)_name", "mode"]].sample(10, random_state=42)

,track_name,artist(s)_name,mode
676,A Tu Merced,Bad Bunny,Major
195,Have You Ever Seen The Rain?,Creedence Clearwater Revival,Major
713,Thinking with My Dick,"Kevin Gates, Juicy J",Major
717,Bohemian Rhapsody - Remastered 2011,Queen,Minor
12,Flowers,Miley Cyrus,Major
290,Shut Down,BLACKPINK,Major
405,Pink Venom,BLACKPINK,Major
413,PUNTO 40,"Baby Rasta, Rauw Alejandro",Major
134,505,Arctic Monkeys,Major
462,Do They Know It's Christmas? - 1984 Version,Band Aid,Major


**Resultado:** Após a pesquisa, descobriu-se que as 10 amostras coletadas estavam em tom C, confirmando a validade da ideia de preencher os valores faltantes com "C". 

### Tipos incorretos
A coluna `streams` foi lida como texto (`str`), embora devesse ser numérica. Quais valores
impedem a conversão para número?

In [74]:
streams_num = pd.to_numeric(df["streams"], errors="coerce")

problemas = df.loc[streams_num.isna(), ["track_name", "artist(s)_name", "streams"]]
print(f"Valores de streams que não são números: {len(problemas)}")
problemas

Valores de streams que não são números: 1


,track_name,artist(s)_name,streams
574,Love Grows (Where My Rosemary Goes),Edison Lighthouse,BPM110KeyAModeMajorDanceability53Valence75Ener...


**Resultado:** uma única linha (574, "Love Grows", Edison Lighthouse) tem em `streams` um texto
corrompido e o valor real foi perdido. As colunas `in_deezer_playlists` e `in_shazam_charts` também foram lidas como texto, mas não são
serão investigadas pois serão descartadas.

In [75]:
# Testando se o resto da linha também está estragado:
df.loc[574]

track_name                            Love Grows (Where My Rosemary Goes)
artist(s)_name                                          Edison Lighthouse
artist_count                                                            1
released_year                                                        1970
released_month                                                          1
released_day                                                            1
in_spotify_playlists                                                 2877
in_spotify_charts                                                       0
streams                 BPM110KeyAModeMajorDanceability53Valence75Ener...
in_apple_playlists                                                     16
in_apple_charts                                                         0
in_deezer_playlists                                                    54
in_deezer_charts                                                        0
in_shazam_charts                      

**Resultado:** As demais colunas da linha estão íntegras (exceto a data
01/01/1970, suspeita, investigada adiante). Proposta para o notebook 02: converter esse valor
em `NaN` e manter a linha nas análises que não usam `streams`, por exemplo a análise de presença em playlists.

### Datas de lançamento
Três verificações: (1) distribuição dos anos; (2) existência de datas impossíveis;
(3) excesso suspeito de datas em 01/01, que podem ser preenchimento padrão.

In [76]:
decadas = (df["released_year"] // 10) * 10 # Agrupando por décadas
print(decadas.value_counts().sort_index()) # Conta quantas músicas tem em cada década e ordena pela década

recentes = (df["released_year"] >= 2022).sum() # Vai ser usado na hipótese de músicas recentes
print(f"\nMúsicas de 2022 ou 2023: {recentes} ({recentes / len(df):.1%})")

released_year
1930      1
1940      2
1950      9
1960      4
1970      7
1980     12
1990     14
2000     20
2010    151
2020    733
Name: count, dtype: int64

Músicas de 2022 ou 2023: 577 (60.5%)


**Resultado:** 77% das músicas são dos anos 2020 e 60,5% (577) são de 2022–2023, amostra
disponível para a Hipótese 3. Há 69 músicas anteriores a 2010 (clássicos ainda muito ouvidos).
**Cuidado para a H3:** 2023 está incompleto no dataset, e músicas mais antigas tiveram mais
tempo para acumular streams (viés de tempo de exposição). Avaliar restringir a H3 a 2022.

In [77]:
datas = pd.to_datetime(
    df[["released_year", "released_month", "released_day"]].rename(columns={
        "released_year": "year",
        "released_month": "month",
        "released_day": "day",
    }),
    errors="coerce",
)
print(f"Datas impossíveis (ex.: 31/02): {datas.isna().sum()}")

Datas impossíveis (ex.: 31/02): 0


In [78]:
contagem_dias = (
    df.groupby(["released_month", "released_day"]) # Agrupando por dia e mês
      .size()
      .sort_values(ascending=False)
)
print(f"Média de músicas por dia do ano: {len(df) / 365:.1f}")
contagem_dias.head(10) # Os 10 dias com mais lançamentos

Média de músicas por dia do ano: 2.6


released_month  released_day
1               1               65
5               6               27
                13              24
1               7               19
10              21              18
12              9               18
11              4               16
5               20              15
6               2               15
3               3               13
dtype: int64

In [79]:
df[(df["released_month"] == 5) & (df["released_day"] == 6)]["artist(s)_name"].value_counts().head()

artist(s)_name
Bad Bunny                      15
Chencho Corleone, Bad Bunny     1
Bomba Est��reo, Bad B           1
Morgan Wallen                   1
Bad Bunny, Jhay Cortez          1
Name: count, dtype: int64

**Resultado:** nenhuma data impossível. A data 01/01 concentra 65 músicas, cerca de 25 vezes a
média esperada (2,6 por dia), o que indica valor de preenchimento padrão. Os demais picos
correspondem a lançamentos de álbuns (ex.: 06/05/2022, *Un Verano Sin Ti*, Bad Bunny, com 15
faixas), portanto são legítimos.

In [80]:
jan1 = df[(df["released_month"] == 1) & (df["released_day"] == 1)]
jan1[["track_name", "artist(s)_name", "released_year"]].sort_values("released_year")

,track_name,artist(s)_name,released_year
439,Agudo M��gi,"Styrx, utku INC, Thezth",1930
469,White Christmas,"Bing Crosby, John Scott Trotter & His Orchestr...",1942
466,Let It Snow! Let It Snow! Let It Snow!,"Frank Sinatra, B. Swanson Quartet",1950
459,A Holly Jolly Christmas - Single Version,Burl Ives,1952
444,Jingle Bell Rock,Bobby Helms,1957
...,...,...,...
395,Space Song,Beach House,2015
138,Perfect,Ed Sheeran,2017
449,Snowman,Sia,2017
610,Astronaut In The Ocean,Masked Wolf,2019


In [81]:
print((jan1["released_year"] >= 2022).sum())

0


**Resultado:** as 65 músicas com data 01/01 são todas de 2020 ou antes, portanto a Hipótese 3 (2022–2023) não é afetada. Para as
análises por mês, 01/01 deve ser tratado como data desconhecida.

### Duplicatas
Existem linhas repetidas? Verificamos linhas totalmente idênticas e linhas com a mesma
combinação de música e artista.

In [82]:
print(f"Linhas totalmente idênticas: {df.duplicated().sum()}")

chave = ["track_name", "artist(s)_name"]
repetidas = df[df.duplicated(subset=chave, keep=False)].sort_values(chave)
print(f"Linhas com mesma música e artista: {len(repetidas)}")
repetidas[chave + ["released_year", "streams", "in_spotify_playlists", "bpm"]]

Linhas totalmente idênticas: 0
Linhas com mesma música e artista: 8


,track_name,artist(s)_name,released_year,streams,in_spotify_playlists,bpm
372,About Damn Time,Lizzo,2022,723894473,2332,109
764,About Damn Time,Lizzo,2022,723894473,9021,109
178,SNAP,Rosa Linn,2022,726307468,3202,170
873,SNAP,Rosa Linn,2022,711366595,1818,170
345,SPIT IN MY FACE!,ThxSoMch,2022,303216294,629,94
482,SPIT IN MY FACE!,ThxSoMch,2022,301869854,573,166
512,Take My Breath,The Weeknd,2021,130655803,2597,121
616,Take My Breath,The Weeknd,2021,432702334,6392,121


In [83]:
repetidas.T # Visualizar melhor

,372,764,178,873,345,482,512,616
track_name,About Damn Time,About Damn Time,SNAP,SNAP,SPIT IN MY FACE!,SPIT IN MY FACE!,Take My Breath,Take My Breath
artist(s)_name,Lizzo,Lizzo,Rosa Linn,Rosa Linn,ThxSoMch,ThxSoMch,The Weeknd,The Weeknd
artist_count,1,1,1,1,1,1,1,1
released_year,2022,2022,2022,2022,2022,2022,2021,2021
released_month,7,4,3,3,10,10,8,8
released_day,15,14,19,19,31,31,6,6
in_spotify_playlists,2332,9021,3202,1818,629,573,2597,6392
in_spotify_charts,2,0,18,0,14,0,0,0
streams,723894473,723894473,726307468,711366595,303216294,301869854,130655803,432702334
in_apple_playlists,0,242,148,3,32,1,17,174


**Take My Breath (The Weeknd):** mesma data (06/08/2021) e mesmo bpm (121), mas tom (A# menor x
G# maior), valence (35 x 53) e liveness (26 x 11) diferentes: provavelmente dois arranjos da
mesma música. Os streams também diferem (131 mi x 433 mi). Aplicada a regra geral: manter a
linha com mais `in_spotify_playlists`.

**0's no in_spotify_charts**

In [84]:
zeros = (df["in_spotify_charts"] == 0).sum()
print(f"Músicas fora dos charts (in_spotify_charts = 0): {zeros} ({zeros / len(df):.1%})")

Músicas fora dos charts (in_spotify_charts = 0): 405 (42.5%)


## Resumo do diagnóstico

| # | Problema | Coluna(s) | Extensão | Tratamento proposto (notebook 02) |
|---|----------|-----------|----------|-----------------------------------|
| 1 | Caracteres perdidos na fonte (`�`) | `track_name`, `artist(s)_name` | 70 e 48 linhas | Manter; corrigir manualmente só os artistas mais frequentes; citar como limitação |
| 2 | Nomes de artistas truncados (ex.: "Bad B") | `artist(s)_name` | a medir | Tratar junto com o item 1 na contagem de artistas |
| 3 | Faltantes em `key` correspondem à nota C | `key` | 95 (9,97%) | Preencher com "C" (amostra externa: 10 de 10 em C) |
| 4 | Texto corrompido em `streams` | `streams` | 1 linha (574) | Converter o valor em `NaN`, manter a linha e converter a coluna para numérica |
| 5 | Colunas de outras plataformas (texto e faltantes) | Apple, Deezer, Shazam | 7 colunas | Remover (fora do escopo) |
| 6 | Datas 01/01 de preenchimento | `released_month`, `released_day` | 65 linhas, todas ≤ 2020 | Tratar dia/mês como desconhecidos em análises por mês; não afeta a H3 |
| 7 | Ano incorreto ("Agudo Mági", 1930) | `released_year` | 1 linha | Corrigir ou marcar como inválido |
| 8 | Mesma música e artista repetidos | `track_name`, `artist(s)_name` | 4 pares | Lizzo e SNAP: mesma gravação (áudio idêntico); ThxSoMch: áudio diferente (versões?), streams quase iguais. Manter, em cada par, a linha com mais `in_spotify_playlists` |
| 9 | Muitos zeros e significado ambíguo | `in_spotify_charts` | 405 (42,5%) | Não é erro; definir tratamento e significado na análise |
| 10 | Viés de tempo de exposição em 2023 | datas, `streams` | H3 | Avaliar restringir a H3 a 2022 |
| 11 | Nomes de colunas com símbolos (`(s)`, `%`) | várias | — | Renomear |